# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/summayashaikh079-stack/flyrank-ml-week1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row = one content page (pseudonymized content_id), 32 clients. Metrics are a trailing-90-day snapshot as of one point in time — not a daily time series (30,000 rows, 44 columns).

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, subprocess
if not os.path.isdir("flyrank-ml-week1"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/summayashaikh079-stack/flyrank-ml-week1"], check=True)
os.chdir("flyrank-ml-week1")

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Rows: {len(df)}, Columns: {len(df.columns)}")
dupe_check = df.groupby('content_id').size()
print(f"Rows with duplicate content_id: {(dupe_check > 1).sum()}")
print(f"Unique clients: {df['client_id'].nunique()}")

Rows: 30000, Columns: 44
Rows with duplicate content_id: 0
Unique clients: 32


## 2. Fields: feature / label / context / excluded

Features: content_type, main_intent, word_count, char_count, search_volume, competition, competition_level, cpc, engagement_rate, scroll_rate, ai_traffic_pct, age_tier, freshness_tier, avg_position, impression_tier, position_tier — all knowable before predicting refresh need.
Label/proxy: trend_direction, trend_pct — the refresh-need signal itself, never a feature.
Context: content_id, client_id — pseudonymized IDs, only for grouping/splitting, never for the model to learn from.
Excluded: provider_used, model_used — which AI tool generated the content; not a performance signal, and mostly missing.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
features = ['content_type','main_intent','word_count','char_count','search_volume',
            'competition','competition_level','cpc','engagement_rate','scroll_rate',
            'ai_traffic_pct','age_tier','freshness_tier','avg_position',
            'impression_tier','position_tier']
label_proxy = ['trend_direction','trend_pct']
context = ['content_id','client_id']
excluded = ['provider_used','model_used']

print(f"Features ({len(features)}): {features}")
print(f"Label/proxy ({len(label_proxy)}): {label_proxy}")
print(f"Context ({len(context)}): {context}")
print(f"Excluded ({len(excluded)}): {excluded}")
print(f"Total classified: {len(features)+len(label_proxy)+len(context)+len(excluded)} out of {len(df.columns)}")

Features (16): ['content_type', 'main_intent', 'word_count', 'char_count', 'search_volume', 'competition', 'competition_level', 'cpc', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'age_tier', 'freshness_tier', 'avg_position', 'impression_tier', 'position_tier']
Label/proxy (2): ['trend_direction', 'trend_pct']
Context (2): ['content_id', 'client_id']
Excluded (2): ['provider_used', 'model_used']
Total classified: 22 out of 44


## 3. Verify it with queries (grain, counts, missing values, windows)

Grain check: content_id has zero duplicates (verified above — 0 out of 30,000). Missingness follows content_type (documented gotcha), checked below. No per-row dates exist — this is a single trailing-90-day snapshot, not a daily panel, so there's no separate time window to verify beyond that.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Missingness follows content_type - the known gotcha
print("\nword_count missing % by content_type:")
print(df.groupby('content_type')['word_count'].apply(lambda x: x.isnull().mean()*100).round(1))


word_count missing % by content_type:
content_type
comparison article     0.0
feedly article         0.0
keyword article       28.3
Name: word_count, dtype: float64


## 4. Data limits

This is a single snapshot (trailing-90-day, as of one date) — not a daily panel, so we can't see how metrics moved over time or what happened right before/after a page was actually refreshed. There's no ground-truth "needs refresh" label; trend_direction/trend_pct is our own proxy, so conclusions are decision-support, not certainty. Missingness isn't random — it's concentrated in "keyword article" content_type (28.3% missing word_count), so any model must handle this with a flag, not a blind fill. provider_used/model_used are mostly missing, so we can't reliably study content-generation-method effects with this data.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"avg_position == 0 (means no data, not rank 0): {(df['avg_position']==0).sum()} rows")
print(f"provider_used missing: {df['provider_used'].isnull().sum()} / {len(df)}")
print(f"model_used missing: {df['model_used'].isnull().sum()} / {len(df)}")
print("\nage_tier distribution:")
print(df['age_tier'].value_counts())

avg_position == 0 (means no data, not rank 0): 1205 rows
provider_used missing: 21438 / 30000
model_used missing: 5733 / 30000

age_tier distribution:
age_tier
91-180     11780
181-365    11368
365+        6360
31-90        492
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.